# Phase 1: Invasive Species Detection with YOLO11m (1024x1024)
This notebook trains an Ultralytics **YOLO11m** (medium) object detection model on the aerial drone imagery dataset to detect two critical invasive plant species in Texas:
- **Class 0: `macartney_rose`** (*Rosa bracteata* - Red bounding boxes)
- **Class 1: `chinese_tallow`** (*Triadica sebifera* - Yellow bounding boxes)

### Key Specifications:
1. **Model:** `yolo11m.pt` (Ultralytics YOLO11 medium)
2. **Resolution:** Resized to `1024x1024` pixels (`imgsz=1024`)
3. **Training Data:** Utilizes all available annotated aerial survey frames
4. **Visualizations:**
   - Overlaid ground truth visualization on dataset samples prior to training
   - Side-by-side **Ground Truth vs. Predicted** comparison after model training


## 1. Environment Setup & Dependencies
Install and verify required dependencies including `ultralytics`, `torch`, `opencv-python`, and `matplotlib`.


In [1]:
# Install dependencies if not already installed in the environment
!pip install --quiet ultralytics opencv-python matplotlib pyyaml pillow tqdm

import os
import sys
import glob
import shutil
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml
import torch
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

ModuleNotFoundError: No module named 'numpy'

## 2. Configuration & Paths Setup
Define directories for raw annotated imagery, generated YOLO dataset splits, and class mappings.


In [ ]:
# Configuration paths
PROJECT_ROOT = Path('/home/bitspark/Varun/TEXAS-IDRT-Invasive-Species-Detection')
SOURCE_IMAGES_DIR = PROJECT_ROOT / 'data' / 'images' / 'Annotated' / 'preview'
WORKSPACE_DIR = PROJECT_ROOT / 'training_phase_1'
DATASET_DIR = WORKSPACE_DIR / 'dataset'

IMAGE_SIZE = 1024  # 1024x1024 resolution
CLASSES = ['macartney_rose', 'chinese_tallow']
CLASS_COLORS = {
    0: ('red', '#e74c3c'),     # Macartney Rose
    1: ('yellow', '#f1c40f')   # Chinese Tallow
}

print(f"Source directory exists: {SOURCE_IMAGES_DIR.exists()}")
source_images = sorted(list(SOURCE_IMAGES_DIR.glob('*.jpg')))
print(f"Found {len(source_images)} aerial drone frames in source directory.")


## 3. Dataset Preparation & YOLO Annotation Extraction
The raw aerial frames in `data/images/Annotated/preview` contain ground truth bounding boxes rendered from survey annotations.
We extract the bounding boxes for both classes:
- **Red outlines:** `macartney_rose` (Class 0)
- **Yellow outlines:** `chinese_tallow` (Class 1)

All samples are prepared and indexed into the training dataset with standard YOLO normalized coordinates:
`class_id center_x center_y width height`


In [ ]:
def extract_bounding_boxes_from_preview(img_path, min_box_size=8):
    """Extract ground-truth bounding boxes for Red (Rose) and Yellow (Tallow) outlines."""
    im = Image.open(img_path).convert('RGB')
    w, h = im.size
    pixels = im.load()
    
    yellow_pixels = set()
    red_pixels = set()
    
    for y in range(h):
        for x in range(w):
            r, g, b = pixels[x, y]
            # Yellow outline detection: high R, high G, low B
            if r > 180 and g > 180 and b < 60:
                yellow_pixels.add((x, y))
            # Red outline detection: high R, low G, low B
            elif r > 180 and g < 60 and b < 60:
                red_pixels.add((x, y))
                
    def get_components(mask_pixels):
        visited = set()
        boxes = []
        neighbors = [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]
        for px in mask_pixels:
            if px in visited:
                continue
            q = [px]
            visited.add(px)
            min_x, max_x = px[0], px[0]
            min_y, max_y = px[1], px[1]
            while q:
                cx, cy = q.pop()
                if cx < min_x: min_x = cx
                if cx > max_x: max_x = cx
                if cy < min_y: min_y = cy
                if cy > max_y: max_y = cy
                for dx, dy in neighbors:
                    nx, ny = cx + dx, cy + dy
                    pos = (nx, ny)
                    if pos in mask_pixels and pos not in visited:
                        visited.add(pos)
                        q.append(pos)
            bw = max_x - min_x
            bh = max_y - min_y
            if bw >= min_box_size and bh >= min_box_size:
                boxes.append((min_x, min_y, max_x, max_y))
        return boxes

    rose_boxes = get_components(red_pixels)
    tallow_boxes = get_components(yellow_pixels)
    return rose_boxes, tallow_boxes, (w, h)

# Setup dataset directory structure
train_img_dir = DATASET_DIR / 'images' / 'train'
train_lbl_dir = DATASET_DIR / 'labels' / 'train'
val_img_dir = DATASET_DIR / 'images' / 'val'
val_lbl_dir = DATASET_DIR / 'labels' / 'val'

for d in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Extracting annotations and building dataset...")
total_rose = 0
total_tallow = 0

for img_file in source_images:
    stem = img_file.stem
    rose_boxes, tallow_boxes, (w, h) = extract_bounding_boxes_from_preview(img_file)
    total_rose += len(rose_boxes)
    total_tallow += len(tallow_boxes)
    
    # Write YOLO label format
    yolo_lines = []
    # Class 0: macartney_rose
    for x1, y1, x2, y2 in rose_boxes:
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        cx = (x1 + x2) / 2.0 / w
        cy = (y1 + y2) / 2.0 / h
        yolo_lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        
    # Class 1: chinese_tallow
    for x1, y1, x2, y2 in tallow_boxes:
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        cx = (x1 + x2) / 2.0 / w
        cy = (y1 + y2) / 2.0 / h
        yolo_lines.append(f"1 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        
    # Save image to train & val splits (all samples used for training)
    dst_img_train = train_img_dir / img_file.name
    dst_lbl_train = train_lbl_dir / f"{stem}.txt"
    dst_img_val = val_img_dir / img_file.name
    dst_lbl_val = val_lbl_dir / f"{stem}.txt"
    
    if not dst_img_train.exists():
        shutil.copy2(img_file, dst_img_train)
    if not dst_img_val.exists():
        shutil.copy2(img_file, dst_img_val)
        
    lbl_content = "\n".join(yolo_lines)
    dst_lbl_train.write_text(lbl_content)
    dst_lbl_val.write_text(lbl_content)

print(f"Dataset built successfully at {DATASET_DIR}!")
print(f"Total Images: {len(source_images)}")
print(f"Total Macartney Rose annotations: {total_rose}")
print(f"Total Chinese Tallow annotations: {total_tallow}")


## 4. Generate YOLO Dataset YAML Configuration
Create the dataset descriptor `data.yaml` pointing to the images and classes.


In [ ]:
data_yaml = {
    'path': str(DATASET_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'macartney_rose',
        1: 'chinese_tallow'
    }
}

data_yaml_path = WORKSPACE_DIR / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"Saved dataset configuration to: {data_yaml_path}")
with open(data_yaml_path) as f:
    print(f.read())


## 5. Visualizing Dataset Samples with Overlaid Ground Truth
Display 2 sample images from the prepared dataset with ground truth bounding boxes and species labels overlaid.


In [ ]:
def visualize_dataset_samples(sample_indices=[1, 8]):
    """Visualize sample images with their ground truth YOLO annotations overlaid."""
    fig, axes = plt.subplots(1, len(sample_indices), figsize=(18, 9))
    if len(sample_indices) == 1:
        axes = [axes]
        
    for ax, idx in zip(axes, sample_indices):
        img_path = source_images[idx]
        stem = img_path.stem
        lbl_path = train_lbl_dir / f"{stem}.txt"
        
        im = Image.open(img_path).convert('RGB')
        w, h = im.size
        ax.imshow(im)
        
        counts = {0: 0, 1: 0}
        if lbl_path.exists():
            lines = [l.strip() for l in lbl_path.read_text().splitlines() if l.strip()]
            for line in lines:
                parts = line.split()
                cid = int(parts[0])
                cx, cy, bw, bh = [float(v) for v in parts[1:]]
                
                # Convert normalized coords back to absolute pixels
                x1 = (cx - bw / 2.0) * w
                y1 = (cy - bh / 2.0) * h
                box_w = bw * w
                box_h = bh * h
                
                color_name, hex_color = CLASS_COLORS[cid]
                counts[cid] += 1
                
                rect = patches.Rectangle(
                    (x1, y1), box_w, box_h,
                    linewidth=2.5,
                    edgecolor=hex_color,
                    facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(
                    x1, max(0, y1 - 6),
                    f"{CLASSES[cid]}",
                    color='white',
                    fontsize=8,
                    fontweight='bold',
                    bbox=dict(facecolor=hex_color, alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2')
                )
                
        ax.set_title(
            f"{img_path.name}\nRose: {counts[0]} | Tallow: {counts[1]}",
            fontsize=12, fontweight='bold'
        )
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

# Show 2 samples with ground truth overlays
visualize_dataset_samples(sample_indices=[1, 8])


## 6. Model Training: YOLO11m at 1024x1024 Resolution
Train Ultralytics **YOLO11m** (`yolo11m.pt`) using:
- **Image Size:** `1024x1024` (`imgsz=1024`)
- **Dataset:** All survey frames (utilizing all available data)
- **Optimizer:** `AdamW` (with `lr0=0.0005` for fine-tuning)
- **Batch Size:** `16` (balanced for GPU VRAM stability)


In [ ]:
# Initialize the YOLO11m model
model = YOLO('yolo11m.pt')

# Train configuration
train_results = model.train(
    data=str(data_yaml_path),
    epochs=40,                  # Number of training epochs
    imgsz=IMAGE_SIZE,           # 1024x1024 resize
    batch=32,                    # Batch size for 6GB-8GB VRAM
    optimizer='AdamW',
    lr0=0.0005,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(WORKSPACE_DIR / 'runs'),
    name='yolo11m_1024_all_samples',
    save=True,
    plots=True,
    verbose=True
)

print("Training completed!")


## 7. Model Inference & Visual Comparison: Ground Truth vs. Prediction
After training, compare the ground truth annotations directly with the trained YOLO11m predictions side-by-side on test frames.


In [ ]:
def compare_ground_truth_vs_prediction(model_path, image_path, conf_thresh=0.25):
    """Plot Ground Truth (left) vs YOLO11m Prediction (right) side by side."""
    im = Image.open(image_path).convert('RGB')
    w, h = im.size
    stem = Path(image_path).stem
    lbl_path = val_lbl_dir / f"{stem}.txt"
    
    # 1. Run prediction with trained model at 1024x1024
    trained_model = YOLO(model_path)
    results = trained_model.predict(
        source=str(image_path),
        imgsz=IMAGE_SIZE,
        conf=conf_thresh,
        device=0 if torch.cuda.is_available() else 'cpu',
        verbose=False
    )[0]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 11))
    
    # --- Left: Ground Truth ---
    ax1.imshow(im)
    gt_counts = {0: 0, 1: 0}
    if lbl_path.exists():
        lines = [l.strip() for l in lbl_path.read_text().splitlines() if l.strip()]
        for line in lines:
            parts = line.split()
            cid = int(parts[0])
            cx, cy, bw, bh = [float(v) for v in parts[1:]]
            x1 = (cx - bw / 2.0) * w
            y1 = (cy - bh / 2.0) * h
            box_w = bw * w
            box_h = bh * h
            gt_counts[cid] += 1
            _, hex_color = CLASS_COLORS[cid]
            rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor=hex_color, facecolor='none')
            ax1.add_patch(rect)
            ax1.text(x1, max(0, y1 - 5), CLASSES[cid], color='white', fontsize=8,
                     bbox=dict(facecolor=hex_color, alpha=0.8, edgecolor='none', pad=1))
    ax1.set_title(f"GROUND TRUTH: {Path(image_path).name}\nRose: {gt_counts[0]} | Tallow: {gt_counts[1]}",
                  fontsize=13, fontweight='bold', color='darkgreen')
    ax1.axis('off')
    
    # --- Right: YOLO11m Prediction ---
    ax2.imshow(im)
    pred_counts = {0: 0, 1: 0}
    for box in results.boxes:
        xyxy = box.xyxy[0].cpu().numpy()
        cid = int(box.cls[0].cpu().numpy())
        conf = float(box.conf[0].cpu().numpy())
        pred_counts[cid] = pred_counts.get(cid, 0) + 1
        
        _, hex_color = CLASS_COLORS.get(cid, ('cyan', '#3498db'))
        rect = patches.Rectangle(
            (xyxy[0], xyxy[1]), xyxy[2] - xyxy[0], xyxy[3] - xyxy[1],
            linewidth=2, edgecolor=hex_color, facecolor='none'
        )
        ax2.add_patch(rect)
        label_text = f"{CLASSES[cid]} {conf:.2f}" if cid < len(CLASSES) else f"Class {cid} {conf:.2f}"
        ax2.text(xyxy[0], max(0, xyxy[1] - 5), label_text, color='white', fontsize=8,
                 bbox=dict(facecolor=hex_color, alpha=0.8, edgecolor='none', pad=1))
                 
    ax2.set_title(f"YOLO11m PREDICTION (imgsz={IMAGE_SIZE}, conf={conf_thresh})\nRose: {pred_counts[0]} | Tallow: {pred_counts[1]}",
                  fontsize=13, fontweight='bold', color='darkblue')
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()

# Locate best weights from training run or fallback to last
best_weights = WORKSPACE_DIR / 'runs' / 'yolo11m_1024_all_samples' / 'weights' / 'best.pt'
if not best_weights.exists():
    best_weights = 'yolo11m.pt'  # pretrained fallback if training not yet completed
print(f"Evaluating weights from: {best_weights}")

# Compare on sample images
compare_ground_truth_vs_prediction(best_weights, source_images[1], conf_thresh=0.25)
compare_ground_truth_vs_prediction(best_weights, source_images[8], conf_thresh=0.25)
